# Atividade Prática - Parte 3
### Grupo 2:<br>
Victor Monteiro <br>
Arthur Horta <br>
João Henrique <br>
Pedro Fioravante <br>

### Chamada das Bibliotecas

In [2]:
# Bibliotecas para análise de dados e visualização
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Módulos do Statsmodels para modelagem e diagnósticos
import statsmodels.api as sm
from statsmodels.stats.diagnostic import (
    het_breuschpagan,
    het_white,
    acorr_breusch_godfrey,
    linear_reset
)
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.compat import lzip

# Estatísticas e testes específicos
from scipy.stats import shapiro, kstest


### ETAPA I

#### Modelo de regressão estimado na atividade 2



In [62]:
# 🔹 Leitura e preparo dos dados
# Lê o arquivo CSV com dados interpolados e converte a coluna 'date' para o formato datetime
df = pd.read_csv(Path("Dados") / "Valores Variaveis Interpolado.csv")
df['date'] = pd.to_datetime(df['date'])

# Calcula os retornos logarítmicos percentuais (log-diferença multiplicada por 100)
# Remove valores ausentes gerados pelo shift
data1 = df.set_index('date').apply(lambda x: (np.log(x) - np.log(x.shift(1))) * 100).dropna()

# 🔹 Definição das variáveis para regressão
# Define a variável dependente (preço do petróleo)
y = data1['price_crude_oil']

# Define as variáveis independentes, adicionando constante ao modelo
X = sm.add_constant(data1.drop(columns=['price_crude_oil']))

# Ajusta o modelo de regressão OLS (Mínimos Quadrados Ordinários)
model1 = sm.OLS(y, X).fit()

# 🔹 Salvando a saída do modelo
# Cria o diretório de saída se não existir
saidasc = Path("Saídas/Modelo sem correcoes"); saidasc.mkdir(parents=True, exist_ok=True)

# Salva o resumo do modelo em um arquivo .txt
with open(saidasc / "OLS Regression Results.txt", "w", encoding="utf-8") as f:
    f.write("=== Modelo Sem Correções ===\n\n" + model1.summary().as_text())

# 🔹 Exibição opcional no console
print("\n Modelo Salvo ✅ \n")


 Modelo Salvo ✅ 



#### Testes de Heterocedasticidade e Autocorrelação

In [69]:
# 🔹 Testes de diagnóstico: heterocedasticidade e autocorrelação
bp    = het_breuschpagan(model1.resid, model1.model.exog)
white = het_white(model1.resid, model1.model.exog)
dw    = durbin_watson(model1.resid)

# 🔹 Tabela com resultados
testessc = pd.DataFrame({
    'Teste': ['Breusch-Pagan', 'White', 'Durbin-Watson'],
    'Estatística': [bp[0], white[0], dw],
    'p-valor':     [bp[1], white[1], None],
    'f-valor':     [bp[2], None, None],
    'f p-valor':   [bp[3], None, None]
})

# 🔹 Salva a tabela como imagem PNG (sem exibição no notebook)
fig, ax = plt.subplots(figsize=(10, 1))
ax.axis('off')
tabelasc = ax.table(cellText=testessc.values, colLabels=testessc.columns, loc='center', cellLoc='center')
tabelasc.scale(1, 1.5)
plt.savefig(Path("Saídas/Modelo sem correcoes") / "Resultados Heterocedasticidade e Autocorrelação.png",
            bbox_inches='tight', dpi=300)
plt.close()

print("\n Dados salvos na pasta 'Saídas/Modelo sem correcoes' ✅ \n")


 Dados salvos na pasta 'Saídas/Modelo sem correcoes' ✅ 



#### Teste de Normalidade e Análise Gráfica dos resíduos

In [72]:
# 🔹 Testes de normalidade dos resíduos
jb_test      = jarque_bera(model1.resid)     # Teste de Jarque-Bera (assimetria e curtose)
shapiro_test = shapiro(model1.resid)         # Teste de Shapiro-Wilk (normalidade geral)

# 🔹 Tabela de resultados
test_results1 = pd.DataFrame({
    'Teste': ['Jarque-Bera', 'Shapiro-Wilk'],
    'Estatística': [jb_test[0], shapiro_test[0]],
    'p-valor':     [jb_test[1], shapiro_test[1]]
})

# 🔹 Salva a tabela como imagem PNG (sem exibição)
fig, ax = plt.subplots(figsize=(8, 1))
ax.axis('off')
tabelasc1 = ax.table(cellText=test_results1.values, colLabels=test_results1.columns, loc='center', cellLoc='center')
tabelasc1.scale(1, 1.5)
plt.savefig(Path("Saídas/Modelo sem correcoes") / "Resultados Normalidade.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n Dados salvos na pasta 'Saídas/Modelo sem correcoes' ✅ \n")


 Dados salvos na pasta 'Saídas/Modelo sem correcoes' ✅ 



In [73]:
# 🔹 Gráfico: Resíduos vs Valores Ajustados
plt.figure(figsize=(10, 5))
plt.scatter(model1.fittedvalues, model1.resid, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Valores Ajustados')
plt.ylabel('Resíduos')
plt.title('Resíduos vs Valores Ajustados')
plt.savefig(Path("Saídas/Modelo sem correcoes") / "Resíduos vs valores ajustados.png", bbox_inches='tight', dpi=300)
plt.close()

print("\n Gráfico salvo na pasta 'Saídas/Modelo sem correcoes' ✅ \n")


 Gráfico salvo na pasta 'Saídas/Modelo sem correcoes' ✅ 



In [76]:
# Q-Q plot dos resíduos
fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111)
sm.qqplot(model1.resid, line='45', fit=True, ax=ax)
ax.set_xlabel('Quantis Distribuição Teórica')
ax.set_ylabel('Quantis dos Resíduos do Modelo')
ax.set_title('Q-Q Plot dos Resíduos')
plt_pathsc1 = Path("Saídas/Modelo sem correcoes") / "Q-Q Plot.png"
plt.savefig(plt_pathsc1, bbox_inches='tight', dpi=300)
plt.close()

print("\n Gráfico salvo na pasta 'Saídas/Modelo sem correcoes' ✅ \n")


 Gráfico salvo na pasta 'Saídas/Modelo sem correcoes' ✅ 



In [77]:
# Histograma dos resíduos
plt.figure(figsize=(10, 5))
sns.histplot(model1.resid, kde=True)
plt.title('Distribuição dos Resíduos')
plt.xlabel('Resíduos')
plt_pathsc2 = Path("Saídas/Modelo sem correcoes") / "Histograma dos Resíduos.png"
plt.savefig(plt_pathsc2, bbox_inches='tight', dpi=300)
plt.close()

print("\n Gráfico salvo na pasta 'Saídas/Modelo sem correcoes' ✅ \n")


 Gráfico salvo na pasta 'Saídas/Modelo sem correcoes' ✅ 



### ETAPA II

#### VIF e Número condicional

In [ ]:
# Análise de Multicolinearidade usando VIF, Número Condicional e Matriz de Correlação
print("\n=== Análise de Multicolinearidade ===")

# VIF (Variance Inflation Factor)
vif_data = pd.DataFrame({
    "Variável": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

# Salvando como imagem PNG
fig, ax = plt.subplots(figsize=(4, 6))
ax.axis('off')
tabelasc3 = ax.table(cellText=vif_data.values, colLabels=vif_data.columns, loc='center', cellLoc='center')
tabelasc3.scale(1, 2)  # aumenta a altura das células

img_pathc3 = Path("Saídas/Modelo sem correcoes") / "Resultado VIF.png"
plt.savefig(img_pathc3, bbox_inches='tight', dpi=300)
plt.close()

# Número Condicional
cond_number = np.linalg.cond(X)
print(f"\n- Conditional Number: {cond_number:.2f}\n")
print("\n- Fator de Inflação da Variância (VIF):\n")
display(vif_data.style.hide(axis='index'))


=== Análise de Multicolinearidade ===

- Conditional Number: 1989.34


- Fator de Inflação da Variância (VIF):



Variável,VIF
const,1.011588
price_gold,1.999848
price_palladium,1.470533
price_platinum,1.967563
price_rhodium,1.006902
price_brent_oil,4.461394
price_dow_jones,13.653115
price_ego,2.208254
price_eur/usd,1.004576
price_gdx_etf,3.079762


#### Matriz de Correlações

In [86]:
# Gerando a matriz de correlação
correlation_matrix = data1.corr().round(3)
plt.figure(figsize=(12, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlação das Variáveis')
path_vif= Path("Saídas/Modelo sem correcoes") / "Matriz de Correlações.png"
plt.savefig (path_vif, bbox_inches= 'tight', dpi=300)
plt.close()

print("\n Matriz salva na pasta 'Saídas/Modelo sem correcoes' ✅ \n")


 Matriz salva na pasta 'Saídas/Modelo sem correcoes' ✅ 



#### Teste de Forma Funcional

In [12]:
# Teste RESET de Ramsey para a forma funcional do modelo
print("\n=== Teste RESET ===")
reset_test = linear_reset(model1, power=3)
print(f"- Estatística F: {reset_test.statistic:.4f}")
print(f"- p-valor: {reset_test.pvalue:.4f}")


=== Teste RESET ===
- Estatística F: 0.2999
- p-valor: 0.8608


#### Aplicando soluções (Stepwise para multicolinearidade e padrão robusto para heterocedasticidade)

##### Regressão Stepwise

In [87]:
# Separando as variáveis dependentes e independentes
y1 = data1['price_crude_oil']
X1 = data1.drop(columns=['price_crude_oil'])

# Função para realizar a regressão stepwise
def stepwise_selection(X1, y1, threshold_in=0.05, threshold_out=0.1):
    """
    Realiza seleção de variáveis utilizando o método stepwise.
    No método stepwise, variáveis são adicionadas e removidas iterativamente
    para encontrar o melhor conjunto de variáveis explicativas com base nos p-valores.

    Parâmetros:
    X1 : Variáveis do dataframe data1 exceto price_crude_oil
    y1 : price_crude_oil do dataframe data1
    threshold_in : float, opcional (default=0.05)
        Limite para adicionar uma variável (forward step).
    threshold_out : float, opcional (default=0.1)
        Limite para remover uma variável (backward step).

    Retorna:
    melhores_var : list
        Lista com os nomes das variáveis selecionadas.
    """
    # Lista inicial de variáveis disponíveis
    initial_features = X.columns.tolist()
    # Lista para armazenar as melhores variáveis selecionadas
    melhores_var = []

    while True:
        changed = False

        # Forward step: adiciona variáveis ao modelo se seu p-valor for menor que o limite definido
        remaining_features = list(set(initial_features) - set(melhores_var))
        new_pval = pd.Series(index=remaining_features, dtype=float)
        for new_column in remaining_features:
            # Ajusta o modelo com as variáveis já selecionadas mais a nova variável candidata
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[melhores_var + [new_column]]))).fit()
            # Armazena o p-valor da nova variável
            new_pval[new_column] = model.pvalues[new_column]

        # Verifica se alguma variável deve ser adicionada
        if not new_pval.empty:
            best_pval = new_pval.min()
            if best_pval < threshold_in:
                # Adiciona a variável com menor p-valor se for estatisticamente significativa
                melhores_var.append(new_pval.idxmin())
                changed = True

        # Backward step: remove variáveis do modelo se seu p-valor for maior que o limite definido
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[melhores_var]))).fit()
        pvalues = model.pvalues.iloc[1:]  # Exclui a constante
        worst_pval = pvalues.max()
        if worst_pval > threshold_out:
            # Remove a variável com maior p-valor se não for estatisticamente significativa
            worst_feature = pvalues.idxmax()
            melhores_var.remove(worst_feature)
            changed = True

        # Se nenhuma variável foi adicionada ou removida, encerra o loop
        if not changed:
            break

    return melhores_var

# Aplicando a seleção stepwise para selecionar as melhores variáveis explicativas
selected_features = stepwise_selection(X1, y1)

# Reajustando o modelo com as variáveis selecionadas e usando matriz de variância-covariância robusta (HC0) para lidar com heterocedasticidade
print("\nVariáveis selecionadas no Stepwise:\n")
print(selected_features)
X_selected = X1[selected_features]
X_selected = sm.add_constant(X_selected)
model2 = sm.OLS(y1, X_selected).fit(cov_type='HC0')

# Salvando a saída
saidac = Path("Saídas/Modelo Corrigido"); saidac.mkdir(parents=True, exist_ok=True)
with open(saidac / "OLS Regression Results.txt", "w", encoding="utf-8") as f:
    f.write("=== Modelo Sem Correções ===\n\n" + model2.summary().as_text())

# Exibindo o resumo do modelo
print("\n === Modelo com Correções ===\n")
print(model2.summary())


Variáveis selecionadas no Stepwise:

['price_brent_oil', 'price_uso_etf', 'price_sp500', 'price_us dollar_index']

 === Modelo com Correções ===

                            OLS Regression Results                            
Dep. Variable:        price_crude_oil   R-squared:                       0.887
Model:                            OLS   Adj. R-squared:                  0.887
Method:                 Least Squares   F-statistic:                     1934.
Date:                Thu, 15 May 2025   Prob (F-statistic):               0.00
Time:                        13:40:22   Log-Likelihood:                -2009.3
No. Observations:                2258   AIC:                             4029.
Df Residuals:                    2253   BIC:                             4057.
Df Model:                           4                                         
Covariance Type:                  HC0                                         
                            coef    std err          z      P>|

#### Testes de multicolinearida, forma Funcional, Heterocedasticidade e Autocorelação do novo modelo

In [ ]:
# Número Condicional
cond_number = np.linalg.cond(X_selected)
print(f"\n- Conditional Number: {cond_number:.2f}\n")

# VIF (Variance Inflation Factor)
print("\n- Fator de Inflação da Variância (VIF):\n")
vif_data1 = pd.DataFrame({
    "Variável": X_selected.columns,
    "VIF": [variance_inflation_factor(X_selected.values, i) for i in range(X_selected.shape[1])]
})

# Salvando como imagem PNG
fig, ax = plt.subplots(figsize=(4, 2))
ax.axis('off')
tabelasc4 = ax.table(cellText=vif_data1.values, colLabels=vif_data1.columns, loc='center', cellLoc='center')
tabelasc4.scale(1, 2)  # aumenta a altura das células

img_pathc4 = Path("Saídas/Modelo Corrigido") / "Resultado VIF.png"
plt.savefig(img_pathc4, bbox_inches='tight', dpi=300)
plt.close()

display(vif_data1.style.hide(axis='index'))


- Conditional Number: 6.08


- Fator de Inflação da Variância (VIF):



Variável,VIF
const,1.006670
price_brent_oil,4.415191
price_uso_etf,4.567091
price_sp500,1.147290
price_us dollar_index,1.025297



 Tabela salva na pasta 'Saídas/Modelo Corrigido' ✅ 


 Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ 



In [94]:
# Teste RESET para verificar se a forma funcional do modelo é adequada
reset_test = linear_reset(model2, power=3)
reset_statistic = reset_test.statistic
reset_pvalue = reset_test.pvalue

# Teste de Heterocedasticidade de Breusch-Pagan
bp_test = het_breuschpagan(model2.resid, model2.model.exog)
bp_labels = ['Lagrange Multiplier statistic', 'p-value', 'f-value', 'f p-value']

# Teste de Heterocedasticidade de White
white_test = het_white(model2.resid, model2.model.exog)

# Teste de Autocorrelação de Durbin-Watson
dw_statistic = durbin_watson(model2.resid)

# Criar DataFrame para a tabela de resultados dos testes estatísticos
test_results2 = pd.DataFrame({
    'Teste': ['RESET', 'Breusch-Pagan', 'White', 'Durbin-Watson'],
    'Estatística': [
        round(reset_statistic, 5),
        round(bp_test[0], 5),
        round(white_test[0], 5),
        round(dw_statistic, 5)
    ],
    'p-valor': [
        round(reset_pvalue, 5),
        round(bp_test[1], 5),
        round(white_test[1], 5),
        None
    ],
    'f-valor': [
        None,
        round(bp_test[2], 5),
        None,
        None
    ],
    'f p-valor': [
        None,
        round(bp_test[3], 5),
        None,
        None
    ]
})

# Salvando como imagem PNG
fig, ax = plt.subplots(figsize=(7,2))
ax.axis('off')
tabelasc5 = ax.table(cellText=test_results2.values, colLabels=test_results2.columns, loc='center', cellLoc='center')
tabelasc5.scale(1, 2)  # aumenta a altura das células

img_pathc5 = Path("Saídas/Modelo Corrigido") / "Resultado Testes.png"
plt.savefig(img_pathc5, bbox_inches='tight', dpi=300)
plt.close()

# Gráfico de Resíduos vs. Valores Ajustados
plt.figure(figsize=(10, 6))
plt.scatter(model2.fittedvalues, model2.resid)
plt.axhline(0, color='red', linestyle='--')
plt.title('Resíduos vs. Valores Ajustados')
plt.xlabel('Valores Ajustados')
plt.ylabel('Resíduos')
plt.savefig (Path("Saídas/Modelo Corrigido") / "Gráfico de Resíduos vs Valores Justados", bbox_inches= 'tight', dpi=300)
plt.close()

print("\n Tabela salva na pasta 'Saídas/Modelo Corrigido' ✅ \n")
print("\n Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ \n")


 Tabela salva na pasta 'Saídas/Modelo Corrigido' ✅ 


 Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ 



### Etapa III

#### Testes de normalidade

In [95]:
# Teste de Jarque-Bera para detectar normalidade nos resíduos
jb_test1 = jarque_bera(model2.resid)

# Teste de Shapiro-Wilk para verificar a normalidade dos resíduos
shapiro_test1 = shapiro(model2.resid)

# Criando DataFrame para a tabela de resultados dos testes
test_results2 = pd.DataFrame({
    'Teste': ['Jarque-Bera', 'Shapiro-Wilk'],
    'Estatística': [jb_test1[0], shapiro_test1[0]],
    'p-valor': [jb_test1[1], shapiro_test1[1]]
})

# Salvando como imagem PNG
fig, ax = plt.subplots(figsize=(7,1))
ax.axis('off')
tabelasc6 = ax.table(cellText=test_results2.values, colLabels=test_results2.columns, loc='center', cellLoc='center')
tabelasc6.scale(1, 2)  # aumenta a altura das células

img_pathc6 = Path("Saídas/Modelo Corrigido") / "Resultado Normalidade.png"
plt.savefig(img_pathc6, bbox_inches='tight', dpi=300)
plt.close()

print("\n Tabela salva na pasta 'Saídas/Modelo Corrigido' ✅ \n")


 Tabela salva na pasta 'Saídas/Modelo Corrigido' ✅ 



In [97]:
# Histograma dos resíduos para verificar a normalidade
plt.figure(figsize=(10, 5))
sns.histplot(model2.resid, kde=True)
plt.title('Distribuição dos Resíduos')
plt.xlabel('Resíduos')
plt.savefig (Path("Saídas/Modelo Corrigido") / "Histograma dos Resíduos", bbox_inches= 'tight', dpi=300)
plt.close()

print("\n Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ \n")


 Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ 



In [99]:
# Q-Q plot para verificar a normalidade dos resíduos
fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111)
sm.qqplot(model2.resid, line='45', fit=True, ax=ax)
ax.set_xlabel('Quantis Distribuição Teórica')
ax.set_ylabel('Quantis dos Resíduos do Modelo')
ax.set_title('Q-Q Plot dos Resíduos')
plt.savefig (Path("Saídas/Modelo Corrigido") / "Q-Q Plot", bbox_inches= 'tight', dpi=300)
plt.close()

print("\n Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ \n")



 Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ 



#### Investigação de Outliers nos Resíduos

In [41]:
# Número de observações e variáveis
n = X1.shape[0]
p = X1.shape[1]

# Limiares para identificar outliers:
cook_threshold = 4 / n

# DFFIT: Usando 2 * sqrt(p/n)
dffit_threshold = 2 * np.sqrt(p / n)

# Obtém as medidas de influência do modelo ajustado
influence = model2.get_influence()

# Extraindo a distância de Cook
cooks_distance = influence.cooks_distance[0] # extrai a distância de Cook

# Extraindo o DFFITS
dffits = influence.dffits[0] # extrai o DFFITS

# Extraindo o DFBETA
dfbeta = influence.dfbetas # extrai o DFBETA

# Extraindo os resíduos studentizados
studres = influence.resid_studentized_internal

# Destacando outliers com base nos limiares
outliers_cooks = np.where(cooks_distance > cook_threshold)[0]
outliers_dffits = np.where(np.abs(dffits) > dffit_threshold)[0]
outliers_dfbetas = np.where(np.abs(dfbeta) > 2)[0]
outliers_studres = np.where(np.abs(studres) > 3)[0]

# Mostrar o número total de outliers
print("Número total de outliers com base na Distância de Cook:", len(outliers_cooks))
print("Número total de outliers com base no DFFIT:", len(outliers_dffits))
print("Número total de outliers com base no DFBETA:", len(outliers_dfbetas))
print("Número total de outliers com base no StudRes:", len(outliers_studres))


Número total de outliers com base na Distância de Cook: 131
Número total de outliers com base no DFFIT: 59
Número total de outliers com base no DFBETA: 0
Número total de outliers com base no StudRes: 40


In [ ]:
# Criando uma figura para o gráfico da Distância de Cook
plt.figure(figsize=(14, 8))
plt.scatter(range(n), cooks_distance, color='blue', s=20, alpha=0.4, label='Distância de Cook')
plt.axhline(y=cook_threshold, color='r', linestyle='--', linewidth=1.25, label='Limiar de Cook')
plt.scatter(outliers_cooks, cooks_distance[outliers_cooks], color='red', s=40, edgecolor='k', label='Outliers de Cook')
plt.xlabel('Observações', fontsize=10)
plt.ylabel('Distância de Cook', fontsize=10)
plt.yscale('log')  # Aplicando escala logarítmica ao eixo y para facilitar visualização
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.legend(fontsize=10)
plt.title('Distância de Cook para cada Observação', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig (Path("Saídas/Modelo Corrigido") / "Distância de Cook", bbox_inches= 'tight', dpi=300)
plt.close()

print("\n Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ \n")

=== Distância de Cook ===


 Gráfico salvo na pasta 'Saídas/Modelo Corrigido' ✅ 

